# Notebook 14 — saved routing-capacity screening results

Purpose: inspect the completed two-hour JEPA screening experiment. Inputs: saved source-verified analysis and figures. Outputs: the predefined recommendation, all encoder scores, paired comparisons and figures in this notebook. Runtime: under one minute. Run All is read-only and never starts training or evaluation.

Default run: `routing-capacity-001`. Override `FMM14_REVIEW_RUN_ID` or `FMM14_REVIEW_ARTIFACT` to view another saved run. No checkpoints are created here. Resume: rerun this notebook to refresh saved report outputs. The original run clock is closed and its FAILED status is preserved. All registered endpoints were completed in a separate recovery phase within the unused execution allowance. This completed run has no remaining training to resume. Checkpoints remain under `checkpoints/routing-capacity-001/`; a future experiment requires its own registered run.

Four paired seeds provide developmental screening, not robust statistical significance. The development images are reused; the official test remains untouched. Missing/cutoff outcomes remain UNKNOWN.

In [ ]:
from pathlib import Path
import os, json, types
from IPython.display import display, Markdown, Image
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
RUN_ID=os.environ.get("FMM14_REVIEW_RUN_ID","routing-capacity-001")
ARTIFACT=Path(os.environ.get("FMM14_REVIEW_ARTIFACT",str(ROOT/"artifacts/notebook_14"/RUN_ID))).resolve()
ANALYSIS=types.ModuleType("fmm14_saved_review");ANALYSIS.__file__=str(ROOT/"experiments/14_routing_capacity/analysis.ipynb")
for cell in json.loads(Path(ANALYSIS.__file__).read_text())["cells"]:
    if cell["cell_type"]=="code" and "ahr-export" in cell.get("metadata",{}).get("tags",[]):
        exec(compile("".join(cell["source"]),ANALYSIS.__file__+"#ahr-export","exec"),ANALYSIS.__dict__)
review=ANALYSIS.load_review(ROOT,ARTIFACT)
display(Markdown("**Saved report status:** "+review["status"]+". Run All only reads saved evidence."))

## Completed screening decision

In [ ]:
if review.get("status")=="COMPLETE":
    analysis=ANALYSIS.read_json(review["analysis_results"])
    if analysis.get("fixture_source"):display(Markdown("**SOFTWARE FIXTURE — NOT TRAINING RESULTS.** "+analysis["fixture_source"]))
    display(Markdown("**Screening decision:** "+analysis["screening_decision"]+". This is a recommendation for further testing, not statistically confirmed superiority. No follow-up is launched."))
    display(Markdown("Four fresh paired seeds; "+str(analysis["updates"])+" full JEPA updates per arm; final fixed-epoch-20 frozen EMA probes. Every condition uses the same reused 2,020 development images. Official test: NOT_RUN. Own-initialization learning: "+analysis["own_initialization_learning"]+"."))
    provenance=analysis.get("completion_provenance")
    if provenance:
        display(Markdown("**Completion history:** The original attempt failed while publishing evaluation outputs. A separate next-day recovery resumed the last 47 training updates and completed all 16 registered evaluations within the unused time allowance. The original FAILED status and clock are preserved; scientific code, schedule, seeds and conditions are unchanged."))
    if any(row.get("median_peak_training_GiB") is None for row in analysis["scores"]):
        display(Markdown("**Cost limitation:** One resumed wide-kernel training arm lacks a recorded peak-memory measurement. Its aggregate memory and cost gate remain UNKNOWN; no missing value is estimated. This does not change the failed accuracy gate."))
    import pandas as pd
    scores=pd.DataFrame(analysis["scores"]);scores["encoder"]=scores["variant"].map(ANALYSIS.NAMES)
    display(scores[["encoder","final_mean_top1_percent","median_jepa_step_seconds","median_peak_training_GiB","numerical_valid"]].round(4))
    for variant,screen in analysis["screening"].items():
        effect=screen["vs_vit"];p=effect["exploratory_holm_adjusted_p_value"]
        display(Markdown("**"+ANALYSIS.NAMES[variant]+": "+screen["status"]+".** Paired mean vs ViT "+f"{effect['mean_percentage_points']:+.3f} pp; raw95% t interval "+ANALYSIS._format_interval(effect["raw_t_ci95_percentage_points"])+"; exploratory Holm p="+str(p)+"; exact sign-flip p="+str(effect["exact_signflip_two_sided_p_value"])+"; positive pairs="+str(effect["positive_pairs"])+"/4. Cost gate: "+screen["cost_gate"]+"."))
    display(Markdown("**Limits:** Four-pair exact two-sided sign-flip tests cannot reach p<.05; raw t inference has df=3 and strong assumptions. New seeds do not turn reused images into independent held-out confirmation. Dense kernels do not demonstrate an FMM speedup."))
else:
    display(Markdown("The complete report is not available. Every declared training/evaluation endpoint must finish before the registered screening decision is calculated. Missing outcomes remain UNKNOWN; rerun this read-only notebook later."))

## Visual evidence

In [ ]:
for figure in review.get("figures",[]):
    display(Markdown("### "+figure["title"]+"\n\n"+figure["note"]))
    display(Image(filename=figure["path"]))
if review.get("html"):
    display(Markdown("Self-contained report: `"+review["html"]+"`. Exact CSV values, calculations and prediction hashes are saved beside it."))